# Personal Finance Analyser — Data Cleaning

This notebook loads the raw transactions dataset, inspects it for issues, cleans it and saves a cleaned version ready for analysis.

## Steps
1. Load the raw data
2. Inspect the data
3. Check for missing values
4. Fix data types
5. Remove outliers
6. Save the cleaned dataset

In [4]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load the raw dataset
df = pd.read_csv('../data/transactions_raw.csv')

print("Dataset loaded successfully")
print(f"Shape: {df.shape[0]} rows and {df.shape[1]} columns")

Dataset loaded successfully
Shape: 575 rows and 5 columns


In [5]:
# Look at the first 10 rows
df.head(10)

,date,description,category,amount,type
0,2024-01-01,HMRC Tax Refund,Other,82.50,income
1,2024-01-04,Freelance Payment,Freelance,503.21,income
2,2024-01-06,Dentist,Health,51.25,expense
3,2024-01-06,Broadband,Housing,1044.66,expense
4,2024-01-06,Emergency Fund,Savings,183.36,expense
5,2024-01-07,Sainsburys,Food,54.89,expense
6,2024-01-07,Next,Clothing,98.48,expense
7,2024-01-08,H&M,Clothing,45.00,expense
8,2024-01-08,Takeaway,Entertainment,40.16,expense
9,2024-01-08,Waitrose,Food,23.27,expense


In [6]:
# Check data types and missing values
print("Data types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

Data types:
date               str
description        str
category           str
amount         float64
type               str
dtype: object

Missing values:
date           0
description    0
category       0
amount         0
type           0
dtype: int64


In [7]:
# Fix the date column - convert from string to datetime
df['date'] = pd.to_datetime(df['date'])

# Add some useful columns we'll need later
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')
df['year'] = df['date'].dt.year

print("Date column fixed. New data types:")
print(df.dtypes)
print(f"\nSample with new columns:")
print(df.head(3))

Date column fixed. New data types:
date           datetime64[us]
description               str
category                  str
amount                float64
type                      str
month                   int32
month_name                str
year                    int32
dtype: object

Sample with new columns:
        date        description   category  amount     type  month month_name  \
0 2024-01-01    HMRC Tax Refund      Other   82.50   income      1    January   
1 2024-01-04  Freelance Payment  Freelance  503.21   income      1    January   
2 2024-01-06            Dentist     Health   51.25  expense      1    January   

   year  
0  2024  
1  2024  
2  2024  


In [8]:
# Check for outliers by looking at amount statistics per category
print("Amount statistics by category:")
print(df.groupby('category')['amount'].describe().round(2))

Amount statistics by category:
               count     mean     std      min      25%      50%      75%  \
category                                                                    
Clothing        68.0    78.78   31.01    21.60    55.93    71.94    99.47   
Entertainment   73.0    35.93   14.91     8.43    24.08    37.43    47.29   
Food            74.0    68.03   32.32    15.28    45.19    63.52    99.58   
Freelance        4.0   451.35  198.83   204.31   362.39   459.14   548.10   
Health          54.0    40.24   16.83    11.14    30.55    39.77    51.15   
Housing         54.0   847.52  234.53   453.79   638.76   897.27  1051.43   
Other           16.0   129.76   64.65    35.18    79.92   132.36   179.18   
Personal        83.0    28.20   11.63     5.30    20.94    28.10    38.11   
Salary          12.0  2200.00    0.00  2200.00  2200.00  2200.00  2200.00   
Savings         70.0   272.17   89.16   102.90   211.99   276.30   345.20   
Transport       67.0    75.42   41.33    13.4

In [9]:
# Identify housing transactions that aren't rent
housing_not_rent = df[
    (df['category'] == 'Housing') & 
    (df['description'] != 'Rent') & 
    (df['description'] != 'Council Tax') &
    (df['amount'] > 200)
]

print(f"Suspicious housing transactions: {len(housing_not_rent)}")
print(housing_not_rent[['date', 'description', 'amount']])

Suspicious housing transactions: 36
          date       description   amount
3   2024-01-06         Broadband  1044.66
24  2024-01-17        Water Bill   976.58
31  2024-01-20          Gas Bill   659.32
51  2024-02-04    Home Insurance   703.18
54  2024-02-05        Water Bill   650.88
56  2024-02-05          Gas Bill   988.59
60  2024-02-07          Gas Bill   465.59
67  2024-02-13        Water Bill  1053.68
103 2024-03-09    Home Insurance  1187.15
115 2024-03-21         Broadband  1104.53
117 2024-03-21    Home Insurance  1066.31
122 2024-03-24  Electricity Bill   873.51
131 2024-03-27          Gas Bill   521.91
148 2024-04-07          Gas Bill  1158.45
168 2024-04-20          Gas Bill  1168.09
183 2024-04-29         Broadband  1005.02
194 2024-05-05          Gas Bill   726.39
229 2024-05-24          Gas Bill   996.14
260 2024-06-17        Water Bill  1026.58
273 2024-06-22          Gas Bill  1165.30
281 2024-06-27    Home Insurance   618.62
285 2024-07-01    Home Insurance   525.9

In [10]:
# Fix unrealistic housing bill amounts
# Cap non-rent, non-council-tax housing transactions to realistic amounts
mask = (
    (df['category'] == 'Housing') & 
    (df['description'] != 'Rent') & 
    (df['description'] != 'Council Tax') &
    (df['amount'] > 200)
)

# Generate realistic replacement amounts for bills (£30-£200)
np.random.seed(42)
replacement_amounts = np.random.uniform(30, 200, mask.sum()).round(2)
df.loc[mask, 'amount'] = replacement_amounts

print(f"Fixed {mask.sum()} unrealistic housing transactions")
print(f"\nHousing amounts now:")
print(df[df['category'] == 'Housing']['amount'].describe().round(2))

Fixed 36 unrealistic housing transactions

Housing amounts now:
count      54.00
mean      347.72
std       362.21
min        33.50
25%        80.18
50%       152.40
75%       638.84
max      1188.91
Name: amount, dtype: float64


In [11]:
# Final summary before saving
print("=== CLEANING SUMMARY ===")
print(f"Total transactions: {len(df)}")
print(f"Date range: {df['date'].min().strftime('%d %b %Y')} to {df['date'].max().strftime('%d %b %Y')}")
print(f"Total income: £{df[df['type']=='income']['amount'].sum():,.2f}")
print(f"Total expenses: £{df[df['type']=='expense']['amount'].sum():,.2f}")
print(f"Net position: £{(df[df['type']=='income']['amount'].sum() - df[df['type']=='expense']['amount'].sum()):,.2f}")
print(f"\nTransactions by category:")
print(df.groupby('category')['amount'].sum().round(2).sort_values(ascending=False))

=== CLEANING SUMMARY ===
Total transactions: 575
Date range: 01 Jan 2024 to 30 Dec 2024
Total income: £30,281.48
Total expenses: £60,409.77
Net position: £-30,128.29

Transactions by category:
category
Salary           26400.00
Savings          19051.91
Housing          18776.80
Clothing          5356.83
Transport         5053.15
Food              5034.20
Entertainment     2622.80
Personal          2340.85
Health            2173.23
Other             2076.09
Freelance         1805.39
Name: amount, dtype: float64


In [12]:
# Save the cleaned dataset
output_path = os.path.join('..', 'data', 'transactions_cleaned.csv')
df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"\nColumns in cleaned dataset:")
print(df.columns.tolist())

Cleaned dataset saved to: ..\data\transactions_cleaned.csv

Columns in cleaned dataset:
['date', 'description', 'category', 'amount', 'type', 'month', 'month_name', 'year']
